# Support Ticket Business Insights

**Project:** AI Customer Support & Resolution Assistant  
**Company:** ServeFlow (fictional)  
**Role:** Business Analyst & Product Manager  
**Dataset:** Synthetic support-ticket dataset  

## Objective
Translate support-ticket data into evidence-backed business insights and product opportunities.

The analysis focuses on ticket volume, resolution performance, escalation, SLA breaches, repeat contacts, and customer satisfaction.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [ ]:
# Load the dataset from the repository structure
df = pd.read_csv('../../data/raw/support_tickets.csv')

df['created_at'] = pd.to_datetime(df['created_at'])

boolean_cols = ['escalated', 'reopened', 'repeat_contact', 'sla_breached']
for col in boolean_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

print(f'Rows: {len(df):,}')
print(f'Columns: {df.shape[1]}')
display(df.head())

## 1. Data Quality Check

In [ ]:
quality = pd.DataFrame({
    'missing_values': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique()
})
display(quality.sort_values('missing_values', ascending=False))

print('Duplicate ticket IDs:', df['ticket_id'].duplicated().sum())

## 2. Executive KPI Baseline

In [ ]:
kpis = pd.Series({
    'Total Tickets': len(df),
    'Avg Resolution Time (hrs)': df['resolution_time_hours'].mean(),
    'Avg First Response (mins)': df['first_response_time_mins'].mean(),
    'Escalation Rate (%)': df['escalated'].mean() * 100,
    'SLA Breach Rate (%)': df['sla_breached'].mean() * 100,
    'Repeat Contact Rate (%)': df['repeat_contact'].mean() * 100,
    'Reopen Rate (%)': df['reopened'].mean() * 100,
    'Average CSAT': df['csat_score'].mean()
})
display(kpis.to_frame('Value'))

## 3. Ticket Volume by Category

In [ ]:
category_volume = df['category'].value_counts().rename_axis('category').reset_index(name='tickets')
category_volume['share_pct'] = category_volume['tickets'] / len(df) * 100
display(category_volume)

category_volume.plot.bar(x='category', y='tickets', legend=False, figsize=(9,5))
plt.title('Ticket Volume by Category')
plt.ylabel('Tickets')
plt.xlabel('Category')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Resolution Performance

In [ ]:
resolution_by_category = (
    df.groupby('category')
      .agg(tickets=('ticket_id','count'), avg_resolution_hours=('resolution_time_hours','mean'), median_resolution_hours=('resolution_time_hours','median'))
      .sort_values('avg_resolution_hours', ascending=False)
)
display(resolution_by_category)

resolution_by_category['avg_resolution_hours'].plot.barh(figsize=(9,5))
plt.title('Average Resolution Time by Category')
plt.xlabel('Hours')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

## 5. Priority and Resolution Time

In [ ]:
priority_order = ['Low', 'Medium', 'High', 'Critical']
priority_perf = (
    df.groupby('priority')
      .agg(tickets=('ticket_id','count'), avg_resolution_hours=('resolution_time_hours','mean'), sla_breach_rate=('sla_breached','mean'), escalation_rate=('escalated','mean'))
      .reindex(priority_order)
)
priority_perf['sla_breach_rate'] *= 100
priority_perf['escalation_rate'] *= 100
display(priority_perf)

## 6. Escalation Analysis

In [ ]:
escalation_by_category = (
    df.groupby('category')
      .agg(tickets=('ticket_id','count'), escalated_tickets=('escalated','sum'), escalation_rate=('escalated','mean'))
      .sort_values('escalation_rate', ascending=False)
)
escalation_by_category['escalation_rate'] *= 100
display(escalation_by_category)

escalation_by_category['escalation_rate'].plot.barh(figsize=(9,5))
plt.title('Escalation Rate by Category')
plt.xlabel('Escalation Rate (%)')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

## 7. SLA Breach Analysis

In [ ]:
sla_by_category = (
    df.groupby('category')
      .agg(tickets=('ticket_id','count'), sla_breaches=('sla_breached','sum'), sla_breach_rate=('sla_breached','mean'))
      .sort_values('sla_breach_rate', ascending=False)
)
sla_by_category['sla_breach_rate'] *= 100
display(sla_by_category)

## 8. Repeat Contacts and Reopened Tickets

In [ ]:
repeat_analysis = (
    df.groupby('category')
      .agg(tickets=('ticket_id','count'), repeat_contact_rate=('repeat_contact','mean'), reopen_rate=('reopened','mean'), avg_csat=('csat_score','mean'))
      .sort_values('repeat_contact_rate', ascending=False)
)
repeat_analysis[['repeat_contact_rate','reopen_rate']] *= 100
display(repeat_analysis)

## 9. Customer Experience

In [ ]:
csat_by_category = (
    df.groupby('category')
      .agg(tickets=('ticket_id','count'), avg_csat=('csat_score','mean'), avg_resolution_hours=('resolution_time_hours','mean'))
      .sort_values('avg_csat')
)
display(csat_by_category)

## 10. Business Insights → Product Opportunities

Use the outputs above to document evidence-backed product implications. The table below is intentionally a template so conclusions come from the observed data rather than assumptions.

In [ ]:
# Automatically surface the highest-risk segments for review
top_resolution = resolution_by_category.head(3)
top_escalation = escalation_by_category.head(3)
top_sla = sla_by_category.head(3)

print('Categories with highest average resolution time:')
display(top_resolution)

print('Categories with highest escalation rate:')
display(top_escalation)

print('Categories with highest SLA breach rate:')
display(top_sla)

### Product implication framework

| Data finding | Business problem | Product opportunity | Candidate AI capability |
|---|---|---|---|
| Highest-resolution-time category | Agents spend more time investigating | Reduce investigation effort | Knowledge retrieval / next-action suggestions |
| Highest-escalation category | More tickets require specialist handoff | Improve handoff quality | Escalation summary |
| Highest-SLA-risk category | Greater operational risk | Identify urgent tickets earlier | Priority recommendation |
| High-volume category | Large support workload | Reduce repetitive triage | Classification + summarization |
| High-repeat-contact category | Customers return for unresolved issues | Improve first-contact resolution | Knowledge assistance |

**Important:** Replace these generic implications with the actual top categories and measured patterns after running the notebook.

## 11. BA/PM Conclusion

The final conclusion should answer three questions:

1. **Where is the largest operational pain?**
2. **Which AI capability directly addresses that pain?**
3. **Which KPI should improve if the capability works?**

The resulting evidence will be used to refine the MVP scope, product metrics, and prototype.